# Threat Level Classification from Crowd Signals

This notebook builds a simple classification pipeline:

- Load the synthetic crowd dataset (or regenerate if not found)
- Create a rule-based `threat_level` label (NORMAL, WARNING, CRITICAL)
- Train a `RandomForestClassifier`
- Save model to `ai/models/rf_threat.pkl`
- Evaluate with confusion matrix and classification report
- Produce a brief textual rationale for a single prediction using feature importances


## Setup

Install dependencies if needed: `pip install scikit-learn pandas numpy joblib seaborn matplotlib`.


In [ ]:
# Imports
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('ai/models', exist_ok=True)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## Load or generate dataset

We first attempt to load a previously generated dataset. If not found, we regenerate a synthetic dataset consistent with the regression notebook and proceed.


In [ ]:
# Try to locate a saved CSV dataset
csv_paths = [
    'ai/data/synthetic_crowd.csv',
    'ai/notebooks/synthetic_crowd.csv'
]

df = None
for p in csv_paths:
    if os.path.exists(p):
        df = pd.read_csv(p, parse_dates=['timestamp'])
        print('Loaded dataset from', p)
        break

if df is None:
    print('No CSV found. Regenerating synthetic dataset (matches the regression notebook logic).')
    # Re-generate a small dataset similar to the regression notebook
    from datetime import datetime, timedelta
    num_sectors = 12
    start_time = datetime(2025, 1, 1, 6, 0, 0)
    num_periods = 24 * 4  # 1 day at 15-minute intervals
    freq_minutes = 15

    rows = []
    for sector in range(num_sectors):
        baseline = np.random.uniform(30, 120)
        supply = np.random.uniform(50, 100)
        for i in range(num_periods):
            ts = start_time + timedelta(minutes=freq_minutes * i)
            hour = ts.hour + ts.minute/60
            cycle = (
                40 * np.sin(2 * np.pi * (hour - 8) / 24.0) +
                30 * np.sin(2 * np.pi * (hour - 18) / 24.0)
            )
            weather_index = 0.5 + 0.5 * np.sin(2 * np.pi * (i / (24 * 4))) + np.random.normal(0, 0.1)
            weather_effect = 10 * weather_index

            inflow = np.clip(np.random.normal(10 + cycle/6 + weather_effect/8, 5), 0, None)
            outflow = np.clip(np.random.normal(8 + cycle/8, 4), 0, None)
            noise = np.random.normal(0, 5)

            current_density = max(0, baseline + inflow - outflow + noise)
            drone_count = int(np.clip(np.round(current_density / 50 + np.random.normal(0, 0.5)), 0, 10))
            alerts_count = int(np.clip(np.round(max(0, np.random.poisson(lam=max(0.2, (current_density - baseline) / 60)))), 0, 20))
            supply = np.clip(supply - current_density * 0.02 + np.random.normal(1.0, 0.5), 0, 100)

            rows.append({
                'timestamp': ts,
                'sector_id': sector,
                'current_density': float(current_density),
                'drone_count': int(drone_count),
                'alerts_count': int(alerts_count),
                'supply_level': float(supply),
                'weather_index': float(weather_index),
            })

    df = pd.DataFrame(rows).sort_values(['sector_id', 'timestamp']).reset_index(drop=True)

print('Shape:', df.shape)
df.head()


## Derive rule-based threat_level

We map `current_density` and `alerts_count` to three levels:

- NORMAL: low density and few alerts
- WARNING: moderate density or some alerts
- CRITICAL: high density and/or many alerts

Thresholds are illustrative; tune per domain knowledge.


In [ ]:
def map_threat_level(density: float, alerts: int) -> str:
    # Simple illustrative rules
    if density >= 140 or alerts >= 6:
        return 'CRITICAL'
    if density >= 90 or alerts >= 3:
        return 'WARNING'
    return 'NORMAL'

thr = df[['current_density', 'alerts_count']].copy()
df['threat_level'] = [map_threat_level(d, a) for d, a in thr.values]

df['threat_level'].value_counts()


## Train classifier and save

We train a `RandomForestClassifier` using a small set of features and save to `ai/models/rf_threat.pkl`.


In [ ]:
feature_cols = [
    'sector_id', 'current_density', 'drone_count', 'alerts_count', 'supply_level', 'weather_index'
]

X = df[feature_cols].copy()
y = df['threat_level'].astype('category')
class_names = list(y.cat.categories)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
clf.fit(X_train, y_train)

model_path = 'ai/models/rf_threat.pkl'
joblib.dump({'model': clf, 'features': feature_cols, 'classes': class_names}, model_path)
print('Saved:', model_path)


## Metrics

We display the confusion matrix and the full classification report on the test set.


In [ ]:
# Predictions and reports
y_pred = clf.predict(X_test)

print('Classification Report:')
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=class_names)
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()


## Simple textual rationale for a single prediction

We show top contributing features via global feature importances and display the top ones for a sample. For more faithful local explanations, consider SHAP or permutation importances.


In [ ]:
# Pick a sample
sample = X_test.iloc[[0]]
pred_class = clf.predict(sample)[0]
proba = clf.predict_proba(sample)[0]

# Global feature importances (simple rationale)
importances = clf.feature_importances_
feat_imp = sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True)

# Textual rationale
top_k = 3
rationale = ", ".join([f"{name}: {imp:.2f}" for name, imp in feat_imp[:top_k]])

print(f"Predicted threat level: {pred_class}")
print(f"Class probabilities: {dict(zip(class_names, np.round(proba, 3)))}")
print(f"Top features contributing globally: {rationale}")
